# pdf Parser

In [ ]:
import fitz

PDF_PATH = r"D:\Ubaid\github\RAGLens\data\thesis.pdf"
pages_text = []

# PyMuPDF - text extraction
with fitz.open(PDF_PATH) as doc:
    total_pages = len(doc)
    for page_num, page in enumerate(doc):
        text = page.get_text("text", sort=True)
        # print(f"Page {page_num + 1}/{total_pages}:\n{text}\n{'-'*40}")
        pages_text.append(text)

In [8]:
def structure_table(raw_table, page_num, table_idx):
    if not raw_table or len(raw_table) < 2:
        return None
    
    # First raw is header
    headers = [str(cell or "").strip() for cell in raw_table[0]]
    rows = []
    for raw_row in raw_table[1:]:
        row = {
            headers[i]: str(cell or "").strip()
            for i, cell in enumerate(raw_row)
            if i < len(headers)
        }
        if any(row.values()):  # skip empty rows
            rows.append(row)

    if not rows:
        return None
    
    # Create a markdown representation for embedding
    md_lines = ["| " + " | ".join(headers) + " |"]
    md_lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        md_lines.append("| " + " | ".join(row.get(h, "") for h in headers) + " |")

    return {
        "page": page_num,
        "table_index": table_idx,
        "headers": headers,
        "rows": rows,
        "row_count": len(rows),
        "col_count": len(headers),
        "markdown": "\n".join(md_lines),
    }


In [9]:
import pdfplumber

PDF_PATH = r"D:\Ubaid\github\RAGLens\data\thesis.pdf"

tables = []

with pdfplumber.open(PDF_PATH) as pdf:
    for page_num, page in enumerate(pdf.pages):
        page_tables = page.extract_tables()
        for table_idx, table in enumerate(page_tables or []):
            if not table:
                continue
            structured_table = structure_table(table, page_num+1, table_idx)
            if structured_table:
                tables.append(structured_table)
                print(f"Page {structured_table['page']} - Table {structured_table['table_index']}:")
                print(structured_table['markdown'])
                print("-" * 40)

# full_text = "\n\n".join(pages_text)



Page 15 - Table 0:
| Parameters | Goal | Acceptable |
| --- | --- | --- |
| PH | 6.5 | 6.5-9.2 |
| Color (Pt-Cu scale) | 15 | 30 |
| Turbidity (NTU) | 5 | 10 |
| Manganese, Mn (mg/lit) | 0.1 | 0.5 |
| Iron, Fe (mg/lit) | 0.3 | 3 |
| Copper, Cu (mg/lit) | 1 | 5 |
| Chloride, Cl (mg/lit) | 250 | 1000 |
| Arsenic, As (mg/lit) | 0.05 | - |
| Cyanide, Cn (mg/lit) | 0.07 | 0.2 |
| Lead, Pb (mg/lit) | 0.01 | 0.1 |
| Mercury, Hg (mg/lit) | 0.001 | 0.002 |
----------------------------------------
Page 19 - Table 0:
|  |
| --- |
| Imaginary Line |
----------------------------------------
Page 20 - Table 0:
| Atomic Weight (12C= 12.0000) | 74.9216 |
| --- | --- |
| Mp at 39.1 Mpa (38.6 atm), ?C | 816 |
| Bp, ?C | 615, sublimes |
| Density at 26?C, Kg/m3 | 5778 |
| Covalent radius | 1.21?A |
| Ionization energy (Kg/mol) | 947 (1st) 1950 (2nd) 2732 (3rd) |
| Latent heat of fusion, J/ (mol K) 2 | 27,740 |
| Latent heat of sublimation, | 31,974 |
| Specific heat at 25?C, µm/(m?C) | 5.6 |
| Electrical

In [ ]:
"""
src/ingestion/parsers/pdf_parser.py
=====================================
PDF parser using PyMuPDF (fast text/image extraction) +
pdfplumber (precision table extraction).

Strategy:
  1. PyMuPDF extracts text page-by-page with layout preservation
  2. pdfplumber extracts tables with row/column structure intact
  3. Combine: replace raw table text with structured table dicts
  4. Tesseract OCR for scanned pages (image-only PDFs)
"""

from __future__ import annotations

import io
from pathlib import Path
from typing import Any

import structlog

from src.core.exceptions import ParserError
from src.core.utils import run_in_threadpool
from src.ingestion.parsers.base import BaseParser, ParsedDocument

logger = structlog.get_logger(__name__)


class PDFParser(BaseParser):
    """
    Production PDF parser combining PyMuPDF + pdfplumber.

    - PyMuPDF: fast, accurate text extraction with page layout
    - pdfplumber: precise table detection and extraction
    - pytesseract: OCR fallback for scanned/image PDFs
    """

    @property
    def supported_extensions(self) -> list[str]:
        return [".pdf"]

    async def parse(self, path: Path) -> ParsedDocument:
        """Parse a PDF file and return structured content."""
        logger.info("pdf_parser_start", path=str(path))
        try:
            result = await run_in_threadpool(self._parse_sync, path)
            logger.info(
                "pdf_parser_done",
                path=str(path),
                pages=result.page_count,
                tables=len(result.tables),
                chars=result.char_count,
            )
            return result
        except Exception as e:
            raise ParserError(f"PDF parsing failed for {path}: {e}") from e

    def _parse_sync(self, path: Path) -> ParsedDocument:
        """Synchronous PDF parsing (runs in thread pool)."""
        import fitz  # PyMuPDF
        import pdfplumber

        pages_text: list[str] = []
        tables: list[dict[str, Any]] = []
        images_ocr: list[dict[str, Any]] = []
        total_pages = 0

        # ── Step 1: PyMuPDF — text extraction ────────────────────────────────
        # with fitz.open(str(path)) as doc:
        #     total_pages = len(doc)
        #     for page_num, page in enumerate(doc):
        #         # Extract text with layout preservation
        #         text = page.get_text("text", sort=True)

        #         # If page has no text (scanned), try OCR
        #         if not text.strip():
        #             text = self._ocr_page(page, page_num)
        #             if text:
        #                 images_ocr.append({"page": page_num + 1, "ocr_text": text})

        #         pages_text.append(f"[Page {page_num + 1}]\n{text}")

        # ── Step 2: pdfplumber — table extraction ─────────────────────────────
        # try:
        #     with pdfplumber.open(str(path)) as pdf:
        #         for page_num, page in enumerate(pdf.pages):
        #             page_tables = page.extract_tables()
        #             for table_idx, raw_table in enumerate(page_tables or []):
        #                 if not raw_table:
        #                     continue
        #                 structured = self._structure_table(
        #                     raw_table, page_num=page_num + 1, table_idx=table_idx
        #                 )
        #                 if structured:
        #                     tables.append(structured)
        # except Exception as e:
        #     # pdfplumber failure is non-fatal — we still have PyMuPDF text
        #     logger.warning("pdfplumber_table_extraction_failed", error=str(e))

        # full_text = "\n\n".join(pages_text)

        # return ParsedDocument(
        #     source_path=str(path),
        #     content=full_text,
        #     doc_type="pdf",
        #     page_count=total_pages,
        #     tables=tables,
        #     images=images_ocr,
        #     metadata={
        #         "file_name": path.name,
        #         "file_size_bytes": path.stat().st_size,
        #         "has_tables": len(tables) > 0,
        #         "has_scanned_pages": len(images_ocr) > 0,
        #     },
        # )

    def _structure_table(
        self,
        raw_table: list[list[str | None]],
        page_num: int,
        table_idx: int,
    ) -> dict[str, Any] | None:
        """Convert raw pdfplumber table to structured dict."""
        if not raw_table or len(raw_table) < 2:
            return None

        # First row is headers
        headers = [str(cell or "").strip() for cell in raw_table[0]]
        rows = []
        for raw_row in raw_table[1:]:
            row = {
                headers[i]: str(cell or "").strip()
                for i, cell in enumerate(raw_row)
                if i < len(headers)
            }
            if any(row.values()):  # skip empty rows
                rows.append(row)

        if not rows:
            return None

        # Also create a markdown representation for embedding
        md_lines = ["| " + " | ".join(headers) + " |"]
        md_lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
        for row in rows:
            md_lines.append("| " + " | ".join(row.get(h, "") for h in headers) + " |")

        return {
            "page": page_num,
            "table_index": table_idx,
            "headers": headers,
            "rows": rows,
            "row_count": len(rows),
            "col_count": len(headers),
            "markdown": "\n".join(md_lines),
        }

    def _ocr_page(self, page: Any, page_num: int) -> str:
        """
        Use Tesseract OCR to extract text from a scanned PDF page.
        Returns empty string if OCR fails or pytesseract is unavailable.
        """
        try:
            import pytesseract
            from PIL import Image

            # Render page to image at 300 DPI
            mat = page.get_pixmap(dpi=300)
            img_data = mat.tobytes("png")
            img = Image.open(io.BytesIO(img_data))
            text = pytesseract.image_to_string(img, config="--psm 3")
            logger.debug("ocr_page", page=page_num + 1, chars=len(text))
            return text
        except Exception as e:
            logger.debug("ocr_failed", page=page_num + 1, error=str(e))
            return ""

In [ ]:
# PDF Parser Implementation
# Strategy:
#   1. PyMuPDF extracts text page-by-page with layout preservation
#   2. pdfplumber extracts tables with row/column structure intact
#   3. Combine: replace raw table text with structured table dicts
#   (Future) Tesseract OCR for scanned pages (image-only PDFs)

import fitz
import pdfplumber
from pathlib import Path
import re


def structure_table(raw_table, page_num, table_idx):
    if not raw_table or len(raw_table) < 2:
        return None
    
    # First raw is header
    headers = [str(cell or "").strip() for cell in raw_table[0]]
    rows = []
    for raw_row in raw_table[1:]:
        row = {
            headers[i]: str(cell or "").strip()
            for i, cell in enumerate(raw_row)
            if i < len(headers)
        }
        if any(row.values()):  # skip empty rows
            rows.append(row)

    if not rows:
        return None
    
    # Create a markdown representation for embedding
    md_lines = ["| " + " | ".join(headers) + " |"]
    md_lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        md_lines.append("| " + " | ".join(row.get(h, "") for h in headers) + " |")

    return {
        "page": page_num,
        "table_index": table_idx,
        "headers": headers,
        "rows": rows,
        "row_count": len(rows),
        "col_count": len(headers),
        "markdown": "\n".join(md_lines),
    }



def parse_pdf(pdf_path):
    pages_text = [] # raw text of each page in order, with layout preserved
    tables = []
    pages = [] # pages with everything structured
    total_pages = 0

    # Extract text with PyMuPDF for better layout preservation
    with fitz.open(pdf_path) as doc:
        total_pages = len(doc)
        for page_num, page in enumerate(doc):
            text = page.get_text("text", sort=True)
            pages_text.append(text)
            pages.append({
                "page_num": page_num + 1,
                "text": text,
                "tables": []
            })

    # Extract tables with pdfplumber for structured data
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            raw_tables = page.extract_tables()
            for table_idx, raw_table in enumerate(raw_tables):
                structured_table = structure_table(raw_table, page_num, table_idx)
                if structured_table:
                    page_text = pages[page_num]["text"]
                    table_title = None
                    for line in page_text.split("\n"):
                        if re.match(r"^\s*table\s*\d*", line.lower()):
                            table_title = line.strip()
                    
                    structured_table["table_title"] = table_title
                    tables.append(structured_table)
                    pages[page_num]["tables"].append(structured_table)

    full_text = "\n\n".join(pages_text)

    return {
        "doc_id": pdf_path.stem,
        "source_path": pdf_path,
        "content": full_text,
        "pages": pages_text,
        "structured_pages": pages,
        "doc_type": "pdf",
        "page_count": total_pages,
        "tables": tables,
        "metadata": {
            "file_name": pdf_path.name,
            "file_size_bytes": pdf_path.stat().st_size,
            "has_tables": len(tables) > 0,
            "has_scanned_pages": False,  # Placeholder for future OCR detection
        }
    }

if __name__ == "__main__":
    PDF_PATH = r"D:\Ubaid\github\RAGLens\data\thesis.pdf"
    PDF_PATH = Path(PDF_PATH)
    result = parse_pdf(PDF_PATH)
    print(f"Parsed PDF: {result['source_path']}")
    print(f"Page Count: {result['page_count']}")
    print(f"Tables Found: {len(result['tables'])}")
    print(f"File Name: {result['metadata']['file_name']}")
    print(f"File Size: {result['metadata']['file_size_bytes']} bytes")
    print(f"Has Tables: {result['metadata']['has_tables']}")
    print(f"Has Scanned Pages: {result['metadata']['has_scanned_pages']}")
    # print a certain page
    n = 15
    # print(f"\n--- Page {n} Text ---")
    # print(result["pages"][n - 1])
    # See any tables found
    from pprint import pprint
    pprint(result["tables"][0])

Parsed PDF: D:\Ubaid\github\RAGLens\data\thesis.pdf
Page Count: 74
Tables Found: 24
File Name: thesis.pdf
File Size: 617808 bytes
Has Tables: True
Has Scanned Pages: False
{'col_count': 3,
 'headers': ['Parameters', 'Goal', 'Acceptable'],
 'markdown': '| Parameters | Goal | Acceptable |\n'
             '| --- | --- | --- |\n'
             '| PH | 6.5 | 6.5-9.2 |\n'
             '| Color (Pt-Cu scale) | 15 | 30 |\n'
             '| Turbidity (NTU) | 5 | 10 |\n'
             '| Manganese, Mn (mg/lit) | 0.1 | 0.5 |\n'
             '| Iron, Fe (mg/lit) | 0.3 | 3 |\n'
             '| Copper, Cu (mg/lit) | 1 | 5 |\n'
             '| Chloride, Cl (mg/lit) | 250 | 1000 |\n'
             '| Arsenic, As (mg/lit) | 0.05 | - |\n'
             '| Cyanide, Cn (mg/lit) | 0.07 | 0.2 |\n'
             '| Lead, Pb (mg/lit) | 0.01 | 0.1 |\n'
             '| Mercury, Hg (mg/lit) | 0.001 | 0.002 |',
 'page': 14,
 'row_count': 11,
 'rows': [{'Acceptable': '6.5-9.2', 'Goal': '6.5', 'Parameters': 'PH'},
     